# 관광 유동인구 이상탐지

| 셀 | 하는 일 |
|---|---|
| 1 | 알고리즘 5종을 **함수**로 구현 |
| 2 | **입력 데이터**를 읽고, 어떤 알고리즘으로 돌릴지 **선택** |
| 3 | 돌려서 **출력** — 알고리즘마다 CSV 한 장 |

출력은 `격자, 시점, 값, 이상여부` 네 열이고 **정상값도 담는다**. 요약 파일은 만들지 않는다.

| 모드 | 알고리즘 | 파일 이름 |
|---|---|---|
| 실제 | 4종 | `grid300m_<알고리즘>_<데이터기준일>.csv` |
| 실제 | `stl` | `grid300m_stl_<시작일>_<종료일>.csv` |
| 예제 | 전부 | `example_<알고리즘>.csv` |

## 1. 알고리즘

In [ ]:
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd


def prepare(x):
    # 숫자로 바꾸고 빈 값을 버린다
    x = pd.Series(x).astype(float)
    return x[~x.isna()]


def finish(x, flag):
    # 모든 알고리즘이 이 모양으로 돌려준다
    return pd.DataFrame({"값": x,
                         "이상여부": pd.Series(flag, index=x.index).astype(bool)})


def iqr(x, k=1.5):
    # 가운데 절반(Q1~Q3)의 폭의 k배를 위아래로 더해 울타리를 친다
    x = prepare(x)
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    spread = q3 - q1
    return finish(x, (x < q1 - k * spread) | (x > q3 + k * spread))


def iqr_log(x, k=1.5):
    # 로그로 큰 값을 눌러 분포를 대칭에 가깝게 만든 뒤 같은 울타리
    x = prepare(x)
    if (x < 0).any():                               # 음수에는 로그를 못 씌운다
        return finish(x, np.zeros(len(x), dtype=bool))
    result = iqr(np.log1p(x), k)
    result["값"] = x                                # 보고는 원래 값으로
    return result


def zscore(x, k=3.0):
    # 평균에서 표준편차 몇 배
    x = prepare(x)
    z = (x - x.mean()) / x.std()
    return finish(x, z.abs() > k)


def mzscore(x, k=3.5):
    # 중앙값에서 MAD 몇 배. 극단값에 끌려가지 않는다.
    x = prepare(x)
    center = x.median()
    spread = (x - center).abs().median()            # MAD
    if spread > 0:
        z = 0.6745 * (x - center) / spread
    else:                                           # MAD가 0인 계열은 분모를 바꾼다
        z = (x - center) / (1.2533 * (x - center).abs().mean())
    return finish(x, z.abs() > k)


def stl(x, period=24, k=3.0, notify=True):
    # 관측값 = 추세 + 주기 + 잔차. 설명되지 않은 잔차가 크면 이상이다.
    x = prepare(x)
    if len(x) < 2 * period:                         # 주기를 두 번은 봐야 분해가 된다
        if notify:
            print("표본 부족 — stl 은 주기의 2배가 필요합니다: %d시점 < %d시점"
                  % (len(x), 2 * period))
        return None

    from statsmodels.tsa.seasonal import STL

    logged = pd.Series(np.log1p(x.to_numpy()))      # 로그 공간에서 분해한다
    resid = pd.Series(np.asarray(STL(logged, period=period, robust=True).fit().resid),
                      index=x.index)
    center = resid.median()
    spread = (resid - center).abs().median()
    if spread > 0:
        z = 0.6745 * (resid - center) / spread
    else:
        z = (resid - center) / (1.2533 * (resid - center).abs().mean())
    return finish(x, z.abs() > k)


ALGORITHMS = {"iqr": iqr, "iqr_log": iqr_log, "zscore": zscore,
              "mzscore": mzscore, "stl": stl}
CHOICES = list(ALGORITHMS) + ["군집분석", "all"]

## 2. 입력 데이터와 알고리즘 선택

경로는 코드에 고정한다 — 이 노트북이 있는 `detection/notebooks/` 에서 센 상대 경로다.
직접 넣어 돌리려면 주석을 풀면 된다.

`stl` 만 입력을 따로 받는다. 한 달치는 격자당 24시점이라 주기의 2배(48시점)에 모자라므로,
`STL_PATTERN` 에 월별 파일이 쌓이면 시점이 이어 붙어 비로소 분해가 된다.

In [ ]:
MODE = "예제"            # "예제" 또는 "실제"
# MODE = input("모드? (예제 / 실제): ").strip() or "예제"

CSV_PATH = Path("../../data"
                "/제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140731.csv")
STL_PATTERN = "../../data/결측치 제거한 데이터/*유동인구_비율포함_20??????.csv"
# CSV_PATH = Path(input("CSV 경로: ").strip())
# STL_PATTERN = input("stl 용 전체기간 경로 또는 글롭: ").strip()

OUT_DIR = Path("out")
PERIOD = 24              # 하루 24시간이 한 주기
PREFIX = "grid300m"      # 출력 파일 이름의 앞머리
HOUR = re.compile(r"^(\d{1,2})시")


def read_table(pattern):
    # 가로형 CSV(한 행 = 격자 하나의 24시간)를 격자 × 시점 표로. 경로 하나든 글롭이든 받는다.
    paths = sorted(glob.glob(str(pattern)))
    if not paths:
        raise FileNotFoundError("해당하는 파일이 없습니다: %s" % pattern)

    frames, days = [], []
    for path in paths:
        for encoding in ("utf-8-sig", "cp949"):     # 파일마다 인코딩이 다르다
            try:
                raw = pd.read_csv(path, encoding=encoding)
                break
            except UnicodeDecodeError:
                continue
        grid_col = [c for c in raw.columns if "격자" in c][0]
        hour_cols = [c for c in raw.columns if HOUR.match(c) and "비율" not in c]
        # 원본에 2014--04 처럼 붙임표가 두 개인 값이 있다 (4·5월). 그대로 두면 달이 뭉친다.
        month = str(raw["기준년월"].iloc[0]).replace("--", "-")[:7]

        long = raw.melt(id_vars=grid_col, value_vars=hour_cols,
                        var_name="열이름", value_name="값")
        long["시점"] = month + "|" + long["열이름"].str[:2]
        long["격자"] = long[grid_col].astype(str).str.replace(r"\.0$", "", regex=True)
        frames.append(long[["격자", "시점", "값"]])
        days.append(str(raw["데이터기준일자"].iloc[0]).replace("-", "")[:8])

    table = (pd.concat(frames).pivot_table(index="격자", columns="시점", values="값")
             .sort_index(axis=1))
    return table, sorted(set(days))


# 손으로 만든 예제 — 답을 알고 있다 (예제 모드의 입력)
series_a = pd.Series([10, 12, 11, 13, 12, 11, 14, 12, 10, 13,
                      90,                                      # t10 — 심은 급증
                      12, 11, 13, 12, 10, 11, 12, 13, 11],
                     index=["t%02d" % i for i in range(20)], dtype=float)

one_day = [12, 10, 9, 9, 10, 13, 18, 26, 38, 46, 52, 58,
           64, 63, 60, 56, 52, 48, 42, 36, 30, 24, 18, 14]
rng = np.random.default_rng(0)
series_b = pd.Series(
    [base * (1 + 0.03 * day) + rng.normal(0, 2.0)
     for day in range(5) for base in one_day],
    index=["D%d|%02d" % (day, hour) for day in range(5) for hour in range(24)])
series_b["D3|03"] = 60.0      # 새벽 급증 — 값 자체는 크지 않다
series_b["D4|13"] = 12.0      # 한낮 급감 — 값 자체는 작지 않다

if MODE == "실제":
    TABLE, DAYS = read_table(CSV_PATH)
    try:
        STL_TABLE, STL_DAYS = read_table(STL_PATTERN)
    except FileNotFoundError:
        STL_TABLE, STL_DAYS = TABLE, DAYS
    before = len(STL_TABLE)
    STL_TABLE = STL_TABLE.dropna()      # 12개월 전부 온전한 격자만 남긴다
    dropped = before - len(STL_TABLE)
else:
    TABLE, DAYS = pd.DataFrame([series_a], index=["예제A"]), []
    STL_TABLE, STL_DAYS = pd.DataFrame([series_b], index=["예제B"]), []
    dropped = 0

print("모드         :", MODE)
print("입력 (4종)   :", TABLE.shape, "격자 × 시점")
print("입력 (stl)   :", STL_TABLE.shape, "격자 × 시점  (결측이 있어 제거한 격자 %s개)"
      % format(dropped, ","))
print("데이터기준일  :", DAYS or "(예제 — 없음)")
print("stl 기간      :", (STL_DAYS[0], STL_DAYS[-1]) if STL_DAYS else "(예제 — 없음)")

try:
    choice = input("어떤 알고리즘으로 이상탐지 할까요?\n  (%s): "
                   % " / ".join(CHOICES)).strip()
except (EOFError, OSError):
    choice = ""
    print("(입력을 받을 수 없는 환경입니다)")
if choice not in CHOICES:
    if choice:
        print("모르는 이름입니다: %r" % choice)
    choice = "all"
print("선택:", choice)

## 3. 출력

알고리즘은 계열 하나만 받는다. 표 전체를 도는 일은 여기서 한다.
`all` 은 알고리즘 **5개**를 돌린다(군집분석은 구현되지 않아 빠진다).
결과가 하나도 없으면 **파일을 만들지 않는다**.

4종은 정상값까지 담는다. **`stl` 만 이상 행만 담는다** — 12개월이면 판정 행이 246만이라
엑셀 한도(1,048,576행)를 넘기 때문이다.

In [ ]:
import time


def run_table(name):
    table = STL_TABLE if name == "stl" else TABLE     # stl 만 전용 입력을 쓴다
    pieces, skipped = [], 0
    for group, series in table.iterrows():
        if name == "stl":
            result = ALGORITHMS[name](series, period=PERIOD, notify=False)
        else:
            result = ALGORITHMS[name](series)
        if result is None:                            # 표본이 모자란 격자
            skipped += 1
            continue
        pieces.append(pd.DataFrame({"격자": group, "시점": result.index,
                                    "값": result["값"].to_numpy(),
                                    "이상여부": result["이상여부"].to_numpy()}))
    if skipped:
        print("  표본 부족으로 건너뛴 격자: %s / %s"
              % (format(skipped, ","), format(len(table), ",")))
    if not pieces:
        return None
    return (pd.concat(pieces, ignore_index=True)
            .sort_values(["격자", "시점"], ignore_index=True))


def output_name(name):
    if MODE == "예제":
        return "example_%s.csv" % name
    if name == "stl":
        days = STL_DAYS or DAYS
        return "%s_stl_%s_%s.csv" % (PREFIX, days[0], days[-1])
    return "%s_%s_%s.csv" % (PREFIX, name, DAYS[-1])


if choice == "군집분석":
    print("군집분석은 아직 구현되지 않았습니다. 파일을 만들지 않습니다.")
    targets = []
else:
    targets = list(ALGORITHMS) if choice == "all" else [choice]

written = []
for name in targets:
    started = time.perf_counter()
    print()
    print("[%s]" % name)
    found = run_table(name)
    if found is None:
        print("  결과가 없습니다 — 파일을 만들지 않습니다 (%.1f초)"
              % (time.perf_counter() - started))
        continue

    anomalies = int(found["이상여부"].sum())
    print("  판정 %s행  이상 %s (%.2f%%)  %.1f초"
          % (format(len(found), ","), format(anomalies, ","),
             100 * anomalies / len(found), time.perf_counter() - started))

    if name == "stl":        # 12개월이면 판정 행이 246만 — 엑셀 한도를 넘어 이상만 담는다
        found = found[found["이상여부"]].reset_index(drop=True)
        print("  stl 은 이상 행만 담습니다 → %s행" % format(len(found), ","))
    if found.empty:
        print("  담을 행이 없습니다 — 파일을 만들지 않습니다")
        continue

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUT_DIR / output_name(name)
    found.to_csv(path, index=False, encoding="utf-8-sig")   # 엑셀용 BOM
    written.append(path)
    print("  저장 %s  (%.1f MB)" % (path, path.stat().st_size / 1024 / 1024))
    if anomalies:
        print(found[found["이상여부"]].head(3).to_string(index=False))

## 4. 군집분석 — 자리

나중에 채운다. 다른 다섯과 달리 **표 전체**를 한꺼번에 받아야 한다.